In [1]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import os

# scifact
# arguana
# nfcorpus
# scidocs
# fiqa

def data_loader(dataset: str):
    url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
    data_path = util.download_and_unzip(url, "data")

    corpus_path = os.path.join(data_path, "corpus.jsonl")
    queries_path = os.path.join(data_path, "queries.jsonl")
    qrels_path = os.path.join(data_path, "qrels", "test.tsv")

    corpus, queries, qrels = GenericDataLoader(
        corpus_file=corpus_path,
        query_file=queries_path,
        qrels_file=qrels_path
    ).load_custom()
    return corpus, queries, qrels


c:\Users\Axel\Desktop\lsa rocchio knn\BEIR_lsa_rocchio_knn\venv\Lib\site-packages\beir\util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "textcat"])

def tokenize_doc(doc):
    return [
        (t.lemma_ or t.text).lower()
        for t in doc
        if t.is_alpha and not t.is_space and not t.is_punct
           and not t.is_stop
           and len(t.text) >= 2
    ]

def clean_data(corpus: dict, queries: dict):
    # Corpus
    ids = list(corpus.keys())
    texts = [(d.get("title","") + " " + d.get("text","")) for d in corpus.values()]
    cleaned_corpus = {}
    for i, doc in zip(ids, nlp.pipe(texts, batch_size=256)):
        cleaned_corpus[i] = {"text": " ".join(tokenize_doc(doc))}

    # Queries
    qids = list(queries.keys())
    qtexts = list(queries.values())
    cleaned_queries = {}
    for qid, doc in zip(qids, nlp.pipe(qtexts, batch_size=256)):
        cleaned_queries[qid] = {"text": " ".join(tokenize_doc(doc))}
    return cleaned_corpus, cleaned_queries


In [3]:
corpus, queries, qrels = data_loader("nfcorpus")
cleaned_corpus, cleaned_queries = clean_data(corpus, queries)

100%|██████████| 3633/3633 [00:00<00:00, 110117.98it/s]


In [4]:
import sys
import os

project_root = r'c:\Users\Axel\Desktop\lsa rocchio knn\BEIR_lsa_rocchio_knn'
sys.path.append(project_root)


In [5]:
vocab_size = len(set(word for doc in cleaned_corpus.values() for word in doc["text"].split()))
print("Taille du vocabulaire :", vocab_size)

Taille du vocabulaire : 19888


In [6]:
from beir.retrieval.evaluation import EvaluateRetrieval

def evaluate_model(model, queries, qrels, k_values=[10, 100]):
    results = {}
    max_k = max(k_values)

    for qid in qrels:
        if qid not in queries:
            continue
        query_text = queries[qid]["text"]
        scores = model.search(query_text, top_k=max_k)
        results[qid] = {doc_id: float(score) for doc_id, score in scores}

    ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels, results, k_values=k_values)
    return (ndcg, _map, recall, precision)



In [7]:
from src.models.bm25 import BM25Model
from src.models.lsa import LSAModel

bm25 = BM25Model()
bm25.fit(cleaned_corpus)
print("bm25 loaded")


lsa = LSAModel()
lsa.fit(cleaned_corpus)
print("lsa loaded")

lsa_rocchio = LSAModel(rocchio_prf=True)
lsa_rocchio.fit(cleaned_corpus)
print("lsa + rocchio loaded")


bm25 loaded
k value: 320
lsa loaded
k value: 320
lsa + rocchio loaded


In [8]:
print(evaluate_model(lsa, cleaned_queries, qrels))
print(evaluate_model(lsa_rocchio, cleaned_queries, qrels))

({'NDCG@10': 0.31292, 'NDCG@100': 0.29604}, {'MAP@10': 0.11707, 'MAP@100': 0.15678}, {'Recall@10': 0.15774, 'Recall@100': 0.30856}, {'P@10': 0.24334, 'P@100': 0.0817})
({'NDCG@10': 0.31023, 'NDCG@100': 0.29659}, {'MAP@10': 0.11274, 'MAP@100': 0.15475}, {'Recall@10': 0.15875, 'Recall@100': 0.31961}, {'P@10': 0.24551, 'P@100': 0.08372})


In [9]:
models = {"bm25":bm25, "lsa":lsa, "lsa_rocchio":lsa_rocchio}
for name, model in models.items():
    res = evaluate_model(model, cleaned_queries, qrels)
    print(f'{name}:\nNDGC@10: {res[0]["NDCG@10"]}, Recall@100: {res[2]["Recall@100"]}')

bm25:
NDGC@10: 0.30367, Recall@100: 0.23548
lsa:
NDGC@10: 0.31292, Recall@100: 0.30856
lsa_rocchio:
NDGC@10: 0.31023, Recall@100: 0.31961
